In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("assignment1.ipynb")

# Assignment 1: 2022 US House Elections <a class="jp-toc-ignore"></a>

## Direction and Goal

![election_prediction](images/clinton-trump-rating.png)  
[image credit](https://donsnotes.com/politics/images/clinton-trump-rating.png)

In this assignment, we'll focus on evaluating the quality of election predictions made by the former political analytics company, [FiveThirtyEight](https://www.natesilver.net/p/a-few-words-about-fivethirtyeight). 

### FiveThirtyEight: Political Data Science Company

FiveThirtyEight pioneered "data journalism" and served as a vital resource for aggregating public polls and rating pollsters. Despite finding a large audience and improving how political data is covered, its parent company, Disney (via ABC News), recently laid off the remaining staff as part of broader cuts. Nate Silver, who founded the site and left two years ago, attributes its decline to Disney's lack of interest in running it as a sustainable business.

### Election Prediction Model

[FiveThirtyEight's House forecast model](https://abcnews.go.com/538/538s-2024-house-election-forecast-works/story?id=114446291) begins by aggregating polling data, such as generic ballot polls and district-specific surveys, to gauge voter sentiment. It then supplements this with a "fundamentals" analysis that predicts results based on factors like partisan lean, incumbency status, and candidate fundraising. To account for qualitative nuances, the model also incorporates expert race ratings from nonpartisan analysts, converting them into numerical inputs. finally, the system simulates various sources of uncertainty—ranging from national shifts to district-specific errors—to produce a probabilistic forecast of the election outcome.

### Evaluating Predictions with Actual Outcomes

We will evaluate US House elections predictions from 2022 found in this [publicly available data](https://github.com/fivethirtyeight/checking-our-work-data/blob/master/us_house_elections.csv). The data contains predictions made by FiveThirtyEight and actual election results from 2022 House elections.

This notebook is based loosely on [this article](https://fivethirtyeight.com/features/the-ncaa-bracket-checking-our-work/). To verify accuracy, the model compares projected win probabilities against actual results: e.g., did roughly 90% of candidates with around 90% win-probability actually win the election?

## Question 1: Data Processing

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">
    
### Q1a: Download Data

Command line interface is a useful tool for programmatically interacting with general functions of your computer: e.g. manipulate/manage files, download from the internet, run scripts, etc.

Below uses command line interace commands to download the raw CSV file from [fivethirtyeight's github page for this data](https://github.com/fivethirtyeight/checking-our-work-data/blob/master/us_house_elections.csv). The `%%bash` at the beginning of a cell tells Jupyter notebook that the cell contains shell commands. 

Ask an LLM to explain the two command line interface commands used above (`wget` and `ls`). Paste the explanation below.

</div>

In [ ]:
%%bash

wget -nc https://raw.githubusercontent.com/fivethirtyeight/checking-our-work-data/master/us_house_elections.csv  -O us_house_elections.csv
ls -l *.csv

**SOLUTION**

<!-- END QUESTION -->

<div class="alert alert-block alert-success">

### Q1b: Load Data

Numpy and Pandas is used to read in the csv file into python.

#### Column Descriptions

The dataset contains the following columns:

**Election Identifiers:**
- `year` - Year of the election (integer)
- `election_date` - Actual date when the election took place (datetime)
- `state` - US state where the race occurred (string)
- `district` - Congressional district number within the state (integer, nullable for at-large districts)
- `office` - Type of office being contested (category, e.g., "U.S. House")

**Forecast Information:**
- `forecast_date` - Date when the prediction was made (datetime)
- `forecast_type` - Type of FiveThirtyEight model used: 'lite', 'classic', or 'deluxe' (category)

**Candidate Information:**
- `candidate` - Name of the candidate (string)
- `party` - Political party affiliation of the candidate (category, e.g., "D" for democratic party and "R" for republican party)

**Prediction & Outcome:**
- `probwin` - Predicted probability that the candidate will win (float, 0-1)
- `probwin_outcome` - Indicator variable for actual election result: 1 if candidate won, 0 if candidate lost (integer)

</div>

In [ ]:
import pandas as pd
import numpy as np

data_q1b = pd.read_csv("us_house_elections.csv")
data_q1b.head()

<div class="alert alert-block alert-success">

#### Data Type Conversions

When reading CSV files, pandas often infers data types automatically, but these may not always be optimal for analysis. Convert the following columns to the appropriate data types:

**Date Columns:**
- `year` - Convert to datetime format using `pd.to_datetime()` with `format='%Y'`, then extract the year as an integer using `.dt.year`
- `election_date` - Convert to datetime using `pd.to_datetime()`
- `forecast_date` - Convert to datetime using `pd.to_datetime()`

**Categorical Columns:** Convert to category type using `.astype('category')` for memory efficiency and faster grouping operations:
- `forecast_type`
- `party`
- `office`

**Numeric Columns:**
- `probwin_outcome` - Convert to `'Int32'`
- `district` - Convert to numeric using `pd.to_numeric()` with `errors='coerce'` to handle non-numeric values, then cast to nullable integer type `'Int64'`

**Text Columns:**
- `candidate` - Convert to string type using `.astype('string')`

After completing the conversions, display the first few rows using `data_q1b.head()` to verify the changes.

</div>

In [ ]:
# Convert columns to appropriate data types
data_q1b['year'] = pd.to_datetime(data_q1b['year'], format= ...
data_q1b['election_date'] = ...
data_q1b['forecast_date'] = ...
data_q1b['forecast_type'] = ...
data_q1b['party'] = ...
data_q1b['probwin_outcome'] = ...
data_q1b['office'] = ...
data_q1b['district'] = pd.to_numeric(data_q1b['district'], errors= ...
data_q1b['candidate'] = ...

In [ ]:
grader.check("q1b")

<div class="alert alert-block alert-success">
    
### Q1c: Subset Data

Fivethirtyeight has three different prediction models: `lite`, `classic` and `deluxe`, which roughly incorporate an increasing number of assumptions.  In this assignment lets focus on evaluting the quality of the `classic` predictions.  You can read more about how their [prediction models work](https://fivethirtyeight.com/methodology/how-fivethirtyeights-house-and-senate-models-work/).

Subset the data by performing the following operations:

1. **Discard 2010 data**: Remove all rows where `year` equals 2010
2. **Keep classic forecast type**: Filter to keep only rows where `forecast_type` is `'classic'`
3. **Keep two forecast dates**: For each election, keep only forecasts made:
   - On election day (`forecast_date` equals `election_date`)
   - 27 days before election day (`forecast_date` equals `election_date - pd.Timedelta(days=27)`)
4. **Sort for comparison**: Sort by `year`, `state`, `district`, `candidate`, and `forecast_date` (all in ascending order) to organize the data for better visual comparison
5. **Reset index**: Reset the DataFrame index after filtering

Save the result back to the variable `data_q1c`.

</div>

In [ ]:
data = (
  data_q1b
  .query("candidate.notnull()")       # remove rows with null candidate names
  .query("...")   # discard 2010 election rows
  .query("...")   # keep only 'classic' forecast type
  .loc[
    (data_q1b.forecast_date == data_q1b.election_date - pd.Timedelta(days=27)) |  # keep forecasts 27 days before election OR
    (...)                                                                 # keep forecasts on election day
    ]
  .sort_values(
    by=[...],       # sort by 'year', 'state', 'district', 'candidate', 'forecast_date'
    ascending=[...] # sort all in ascending order
    )
  .reset_index(drop=True) # reset index after filtering
)
data_q1c['is_election_day'] = (data_q1c['forecast_date'] == data_q1c['election_date'])

In [ ]:
grader.check("q1c")

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">
    
### Q1d: Normalizing Candidate Names

Normalizing candidate names is a significant challenge in data science because the same individual can be recorded in multiple ways across different datasets or even within the same source. Variations arise from the use of middle names or initials, nicknames (e.g., "Alex" vs. "Alexander"), titles (e.g., "Dr.", "Mr."), suffixes (e.g., "Jr.", "III"), and the presence of diacritics or special characters. For example, in this dataset, you might encounter variations like "Mark Sanford" and "Marshall Clement Sanford Jr.", or "Alex X. Mooney" and "Alexander Xavier Mooney".

To address this, we establish a context for name normalization by grouping candidates by their state, district, and party. Within these groups, we apply a name similarity metric—such as those provided by the `rapidfuzz` library—to compare unique name entries pairwise. By calculating similarity scores that account for token ordering and partial matches, we can identify and merge names that exceed a specific threshold (e.g., 80%), ensuring that multiple records for the same candidate are treated as a single entity for analysis.

Regular expressions (regex) and fuzzy string matching are essential tools for data cleaning and entity resolution. While regex allows us to programmatically remove "noise"—such as middle initials, titles (Dr., Mr.), or punctuation—by defining specific patterns, it often struggles with semantic variations like "Alex" vs. "Alexander". 

Fuzzy matching bridges this gap by calculating a similarity score between two strings, allowing us to merge records that are "close enough" even if they aren't identical. 

By combining these tools, we can automate the tedious process of name normalization, ensuring our analysis treats the same individual as a single entity across the entire dataset.

Use LLM chat of your choice to better understand the nuts and bolts of the following functions. Ask the following questions:

1. What does this function do? [Copy and paste code into AI chat]
2. How do `fuzz.token_set_ratio` and `fuzz.partial_token_sort_ratio` differ? When would one score be low and the other be high? Give examples to illustrate their differences.

</div>

In [ ]:
import unicodedata
import re
from rapidfuzz import fuzz

def sanitize_name(name):
  
  name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
  name = re.sub(r'\b\w+\.(?!\S)\s*', '', name).strip()

  return name

def compare_names(n1, n2):

    score1 = fuzz.token_set_ratio(n1, n2, processor=sanitize_name)
    score2 = fuzz.partial_token_sort_ratio(n1, n2, processor=sanitize_name)

    return max(score1, score2)

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<div class="alert alert-block alert-success">

Using `compare_names`, we will identify pairs of similar names and standardize them. Complete the procedure in the next cell by following these steps:

1. **Group by Context**: Group the data by `state`, `district`, and `party`. This serves two purposes:
    - **Efficiency**: Comparing names only within relevant groups is computationally faster than comparing all names in the dataset.
    - **Accuracy**: It prevents false positives by assuming that similar names in different districts (likely) represent different individuals.
2. **Identify Unique Names**: Within each group, extract the unique candidate names (ignoring `NaN` values).
3. **Pairwise Comparison**: Use `itertools.combinations` to generate all possible pairs of names within the group.
4. **Similarity Threshold**: For each pair, calculate the similarity score using `compare_names`. If the score exceeds 80:
    - Identify the **shorter** and **longer** versions of the name.
    - Replace all instances of the longer name with the shorter name in the main `data` DataFrame.
5. **Track Replacements**: Maintain a dictionary to keep track of names that have already been replaced. This ensures that if a name is updated, subsequent comparisons use the most recent version.

Replace redacted code portion denoted by `...` in order to consolidate candidate names.

</div>

In [ ]:
from itertools import combinations

data_q1d = data_q1c.copy()

for ... in data_q1d.groupby(["...", "...", "..."], observed=True):

  # get unique candidate names in the group
  names = group["candidate"].dropna().unique()

  # initialize dictionary to keep track of replaced candidate names
  replaced = dict()

  # iterate over all candidate name pairs to identify similar pairs and replace them
  for n1, n2 in ...(names, 2):

    # n1 and n2 may have been replaced in previous iterations
    # if so, retrieve the latest replaced names
    if n1 in replaced: n1 = replaced[n1]
    if n2 in replaced: n2 = replaced[n2]

    # compare candidate names
    score = ...(n1, n2)

    # replace longer name with shorter name if similarity score is high
    if score > 80:
      shorter = n1 if len(n1) <= len(n2) else n2
      longer = n1 if len(n1) >= len(n2) else n2
      data_q1d['candidate'] = data_q1d['candidate'].replace(..., ...)
      replaced[...] = ...

In [ ]:
grader.check("q1d2")

<div class="alert alert-block alert-success">
    
### Q1e: Filtering Data

In the previous question, we subset the data to include two forecast dates for each election. Ideally, each candidate should have exactly two predictions. However, some candidates may be missing one of these predictions or have invalid name entries.

Filter the `data_q1d` DataFrame to remove any candidate who does not have exactly two predictions. Use a [lambda function](https://www.datacamp.com/tutorial/python-lambda-functions) within a grouping operation to identify these cases.

Save the filtered result as `data_q1e`.

**Key Methods to Use:**
* [`pandas.DataFrame.groupby`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html)
* [`pandas.core.groupby.DataFrameGroupBy.filter`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.core.groupby.DataFrameGroupBy.filter.html)
* [`pandas.DataFrame.shape`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.shape.html) (to verify the result)

Ensure you are using the correct documentation version for your environment (check via `pd.__version__`).

</div>

In [ ]:
data_q1e = data_q1d.groupby([..., ..., ..., ...]).filter(lambda x: ...)

In [ ]:
grader.check("q1e")

## Question 2: Exploratory Analysis

<div class="alert alert-block alert-success">
    
### Q2a: Calculate Change in Win-probability

Let's see if we can find the candidates whose standings change the most between September 19 and November 8: one with largest improvement and another with largest decrease in win-probability. First, use the `agg` function calculate the difference.

Following functions have been useful for me:

* [`numpy.diff`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.diff.html)
* [`pandas.DataFrame.sort_values`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html)
* [`pandas.DataFrame.groupby`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html)
* [`pandas.DataFrame.agg`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.agg.html): especially, [different functions to columns](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#applying-different-functions-to-dataframe-columns) 

Save the resulting DataFrame from `agg()` to a variable, `probwin_change`.

</div>

In [ ]:
probwin_change = data_q1e.groupby([..., ..., ..., ...]).agg({'probwin': lambda x: np.diff(x).item()})

In [ ]:
grader.check("q2a")

<div class="alert alert-block alert-success">
    
### Q2b: Looking for Largest Changes

Identify the candidates who saw the most significant shifts in their win probabilities. Using the `probwin_change` DataFrame, find the candidate with the largest increase and the candidate with the largest decrease in win probability. 

Assign the index of the candidate with the greatest improvement to `rising_candidate_idx` and the candidate with the greatest decline to `falling_candidate_idx`.

</div>

In [ ]:
rising_candidate_idx = ...
falling_candidate_idx = ...

In [ ]:
grader.check("q2b")

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">
    
### Q2c: Verify Outcome

Did the candidate win or lose the election? Verify with election outcome.

</div>

**SOLUTION**

<!-- END QUESTION -->

<div class="alert alert-block alert-success">

### Q2d: State-level Volatility
Political scientists are often interested in identifying which states experienced the most significant shifts in momentum during the final month of a campaign. We can quantify the average absolute change in win probability across all districts within a state for each election year.

Using the `probwin_change` DataFrame, calculate the mean absolute change in win probability for each state, grouped by year. Identify the top 3 state-year combinations with the highest average volatility for each year. 

**Key Steps:**
1. Apply `np.abs()` to the `probwin` column in `probwin_change`.
2. Group by both the `year` and `state` levels of the MultiIndex.
3. Calculate the mean and sort the results in descending order.
4. Keep top three states with largest average absolute changes for each election year.

</a>

In [ ]:
state_volatility = (
  probwin_change['probwin']
  .abs() 
  ...     # groupby or calculate mean?
  ...     # groupby or calculate mean?
  ...     # sort_values or reset_index?
  ...     # sort_values or reset_index?
  .groupby('year')
  .head(3)
  .reset_index(drop=True)
)

In [ ]:
grader.check("q2d")

<div class="alert alert-block alert-success">

## Question 3: Prediction vs Actual Outcomes

We want to check whether events predicted by 538 to occur with probability _close to_ X% actually occurred about X% of the time.

To do this, take a group of candidates and bin them based on `probwin` (forecasted win probability) then compare whether `probwin`-bin is similar to the actual proportion of candidates who won the election.

</div>

<div class="alert alert-block alert-success">
    
### Q3a: Transform Data

In this problem, we will bin each row into 20 separate bins based on `probwin`. Then, we will compute the center of each bin.

</div>

In [ ]:
data_q3a = data_q1e.copy()

bin_ends = np.linspace(..., ..., ...)  # divide into 20 equal-width bins from 0 to 1
bin_ctrs = np.round((... + ...)/2, 3)  # calculate bin centers: (a+b)/2 gives midpoint between a and b

data_q3a['probwin_bin_idx'] = pd.cut(data_q3a[...], ..., include_lowest=..., labels=...) # assign each row a bin index based on probwin
data_q3a['probwin_bin'] = bin_ctrs[data_q3a[...]] # assign each row a bin center based on bin index


In [ ]:
grader.check("q3a")

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">
    
### Q3b: Prediction Histogram

Filter `data_q3a` to include only forecasts made on election day. Then, use `seaborn.displot` to create a histogram of `probwin` values, faceted by `year`. Use `bin_ends` to define the bins.

Your figure will look like this:
![histogram example](images/q3b-histogram.png)

</div>

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.displot(data=..., x="...", col="...", kind="hist", bins=...)
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">
    
### Q3c: Prediction difficulty 

Based on the histograms in Q3b, are most house elections easy to forecast or hard to forecast? State your reasoning.

</div>

**SOLUTION**

<!-- END QUESTION -->

<div class="alert alert-block alert-success">
    
### Q3d: Compute Actual Outcomes

In Q3a, we've grouped the observations into a discrete set of bins according to the predicted probability, `probwin`.  Within each bin, we now want to compute the actual fraction of times the candidates won.

If 538 did a good job, it will be close to the predicted probabilities.  You'll need to use the `groupby` function to compute the mean of `probwin_outcome` (1 is a win and 0 is a loss) within each bin. Once again you can use `agg` method here.

There are three years of election results and two forecasts per candidate. We could ask questions like, are there systematic differences between the years? Is there a difference between 27-day forecast and the-day-of forecast?

Save the fraction of actual wins in each bin in a DataFrame called `data_q3d`.

</div>

In [ ]:
data_q3d = (
  data_q3a
  .groupby([..., ..., ...], observed=True) # For each year, whether forecast was on election day, and probability bin combination,
  .agg(
    sample_size=(..., ...),             # Count number of candidates in each bin
    win_proportion=(..., lambda x: ...) # Calculate win proportion among candidates in each bin
    )
)

In [ ]:
grader.check("q3d")

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">
    
### Q3e: Visualizing Predicted vs. Actual Win-Probability

In this task, we will visualize the calibration of the forecasts by plotting the predicted win probabilities against the actual fraction of wins observed in each bin.

Using the proportions calculated in the previous step, create a visualization with six lines representing each combination of the three election years and the two forecast timings (Election Day and 27 days prior).

</div>

In [ ]:
ax = sns.lineplot(data_q3d, x="probwin_bin", y="win_proportion", hue="year", style="is_election_day", markers=True, dashes=False)
ax.set(
  xlabel="Probability of Win (Binned)", 
  ylabel="Proportion of Candidates Who Won",
  title="Predicted Win-Probability vs Actual Win-Proportion")
plt.plot([0, 1], [0, 1], '--', color='gray')  # reference line y=x;
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">
    
### Q3f: Faceted Visualization

The previous plot is cluttered with many lines, making visual comparisons difficult.

In this question, you will create faceted visualizations using [Seaborn's `relplot` method](https://seaborn.pydata.org/generated/seaborn.relplot.html) to improve clarity.

Generate three sets of visualizations:

1. Plot six combinations in each plot faceting columns by `year` and rows by `is_election_day`. Figures should look like this  
  ![year/electionday](images/q3f-facet-year-electionday.png)
1. Compare the three election years for each forecast timing (Election Day and 27 days prior). Figures should look like this  
  ![year/electionday](images/q3f-facet-year.png)
1. Compare the two forecast dates for each election year. Figures should look like this  
  ![year/electionday](images/q3f-facet-electionday.png)

Comment on any patterns you observe.

</div>

_Type your answer here, replacing this text._

In [ ]:
def annotate_lineplot(ax):
  ax.map(plt.axline, xy1=(0., 0.), slope=1, color='gray', linestyle='--')
  ax.map(plt.hlines, y=[0, 1], xmin=0, xmax=1, color='pink', linestyle=':')
  ax.set(
    xlabel="Probability of Win (Binned)", 
    ylabel="Proportion of Candidates Who Won"
  )

ax = sns.relplot(
  data_q3d, kind="line", col=..., 
  x=..., y=..., hue=...,
)
annotate_lineplot(ax)
plt.show()

In [ ]:
ax = sns.relplot(
  data_q3a, kind="line", col=..., 
  x=..., y=..., hue=...,
)
annotate_lineplot(ax)
plt.show()

In [ ]:
ax = sns.relplot(
  data_q3a, kind="line", col=..., 
  x=..., y=..., hue=...,
)
annotate_lineplot(ax)
plt.show()

<!-- END QUESTION -->

<div class="alert alert-block alert-success">

## Question 4: Quantifying Uncertainty

So far, we computed point estimates of actual winning proportions for each of the twenty groups of similarly forecasted individuals.

In this question we will characterize the uncertainty of the estimated actual winning proportions by using bootstrap.

</div>

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">

### Q4a: Seaborn's Parametric and Nonparametric Error bars

In addition to win proportions point estimate, we want to characterize the uncertainty around it. Using Seaborn package, we can plot parametric and nonparametric error bars.

Create two faceted plots (faceting columns by `year` and rows by `is_election_day`) to put error bars around the winning proportion of candidates in each bin. Read this [Seaborn tutorial on error bars](https://seaborn.pydata.org/tutorial/error_bars.html) to understand the procedure being carried out in order to determine the error bars.

Specifically, you will plot parametric and nonparametric uncertainty error bars (refer to [this table](https://seaborn.pydata.org/tutorial/error_bars.html#statistical-estimation-and-error-bars)).

#### Parametric uncertainty bars

Use `errorbar=("se", 1.96)`. Your figure would look like:

![Parametric uncertainty bars](images/q4a-year-electionday-parametric-se.png)

#### Nonparametric uncertainty bars

Use `errorbar=("ci", 95)` and `n_boot=500`.  Your figure would look like:

![Nonparametric uncertainty bars](images/q4a-year-electionday-nonparametric-ci.png)

</div>

_Type your answer here, replacing this text._

In [ ]:
ax = sns.relplot(
  data_q3a, kind="line", col=..., 
  x=..., y=..., 
  estimator=..., errorbar=...
)
annotate_lineplot(ax)
plt.show()

In [ ]:
ax = sns.relplot(
  data_q3a, kind="line", col=..., 
  x=..., y=..., 
  estimator=..., errorbar=...
)
annotate_lineplot(ax)
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">

### Q4b: Constructing Parametric Error Bars

We will construct parametric error bars for 2018 from our previous result, `data_q3d`. Recall that there are six "histograms", one for each `year` x `is_election_day` combination. Each "histogram" is of predicted win-probability (binned horizontal axis) vs. actual win-proportion (vertical axis).

We will use `year=2018` and `is_election_day=True` histogram in `data_q3d` to compute the parametric uncertainty interval for each bin.

Let's assume whether each candidate in each bin wins behaves like a Bernoulli trial. Denote by $\hat p_b$ the `win_proportion` and $n_b$ denote the `sample_size` of bin $b\in\{0.025, 0.075, \dots, 1\}$. Parametric 95%-confidence interval is
$$\hat p_b \pm 1.96 \cdot\text{SE}_b,$$
where the standard error is defined as
$$\text{SE}_b = \sqrt{\frac{\hat p_b(1-\hat p_b)}{n_b}}$$

If everything went according to the plan, your figure will look like the following (Purple bars are manually computed error bars):

![manual error bar comparison](images/q4b-manual-parametric-se.png)

</div>

In [ ]:
def compute_se(row):

  n = row['sample_size']
  p_hat = row['win_proportion']

  # compute **lengths** of upper and lower bars for 95% confidence
  upper_len = ...
  lower_len = ...

  return pd.Series({'mean': p_hat, 'lower_len': lower_len, 'upper_len': upper_len})

data_q4b = (
  data_q3d
  .query(...)   # filter for year 2018 and election day forecasts   
  .apply(...)   # use compute_se function to calculate mean and error bar lengths
  .reset_index()
)

In [ ]:
# plot Seaborn error bars
ax = sns.relplot(
  data_q3a.query(...),  # filter for year 2018 and election day forecasts
  kind=..., col=..., row=...,
  x=..., y=..., 
  estimator=..., errorbar=...
)

annotate_lineplot(ax)

# overlay manual error bars 
plt.errorbar(data_q4b['probwin_bin'],
             data_q4b['mean'], 
             yerr=data_q4b.loc[:, ['lower_len', 'upper_len']].transpose(),
             fmt='none', elinewidth=1, 
             capsize=3, capthick=1, color='purple')
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">

### Q4c: Constructing Nonparametric Error Bars

In this question, we will use bootstrap to come up with a nonparametric uncertainty interval.

This time, subset `data_q3a` with the same criteria as Q4b. 

Conceptually, nonparametric error bar construction procedure is as follows. For each bin $k$ with $n_k$, perform bootstrap iteration, $b=1,2,\dots,B$, as follows:

1. Resample rows at random from above table $n_k$ times.
2. Calculate mean of `probwin_outcome` on resampled data (call this one bootstrapped mean, $\hat p_k^{(b)}$).
3. Repeat above steps $B$ times, say $B=1000$
4. Return percentile cutoffs at $2.5\% \times 1000$-smallest and $97.5\% \times 1000$-smallest as the bootstrap confidence interval.

Your figure will look like the following (Purple bars are manually computed error bars):

![manual error bar comparison](images/q4c-manual-nonparametric-ci.png)

Note that figure would be similar but not identical due to random nature of bootstrap-based procedures.

</div>

In [ ]:
def bootstrap_ci(df, n_boot=1000, conf=95):
  
  n = len(df)
  boot_means = pd.Series(np.zeros(n_boot)) # initialize series to hold bootstrap means

  for i in range(n_boot):
    sample = df.sample(...)                     # draw bootstrap sample with replacement
    boot_means.iloc[...] = sample[...].mean()   # compute mean of probwin_outcome for bootstrap sample

  mean = ...                        # compute mean of original observations
  lower_limit = np.percentile(...)  # compute lower limit of confidence interval
  upper_limit = np.percentile(...)  # compute upper limit of confidence interval

  output = pd.Series({
    'mean': mean,                     # mean of observations
    'lower_len': mean - lower_limit,  # distance to lower limit 
    'upper_len': upper_limit - mean   # distance to upper limit
    }) 

  # print("n=%3d, idx=%4s, sum=%3d, lower=%.3f, mean=%.3f, upper=%.3f" % (n, df['probwin_bin_idx'].unique(), df['probwin_outcome'].sum(), lower_limit, mean, upper_limit))

  return output

import random

random.seed(10)

data_q4c = (
  data_q3a.query("year == 2018 and is_election_day == True") 
  .groupby('probwin_bin', observed=True)
  .apply(
    bootstrap_ci,   # use the bootstrap_ci function defined above
    n_boot=1000,     # number of bootstrap iterations
    conf=95,        # confidence interval percentage
    include_groups=False
  )
  .reset_index()
)

In [ ]:
# plot Seaborn error bars
ax = sns.relplot(
  data_q3a,
  kind=..., col=..., row=...,
  x=..., y=..., 
  estimator=..., errorbar=..., n_boot=...
)
annotate_lineplot(ax)

# overlay manual error bars 
plt.errorbar(data_q4c['probwin_bin'],
             data_q4c['mean'], 
             yerr=data_q4c.loc[:, ['lower_len', 'upper_len']].transpose(),
             fmt='none', elinewidth=1, 
             capsize=3, capthick=1, color='purple')
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-block alert-success">

### Q4d: Comparing Uncertainty Quantification Approaches

Compare the parametric and non-parametric error bars. What are the advantages and disadvantages of each? Assume extreme situations such as

1. Data is extremely limited vs. large amount of data is available.
2. Assumption about Bernoulli trial is correct vs. not correct.

</div>

_Type your answer here, replacing this text._

<!-- END QUESTION -->



## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(filtering=False, run_tests=True)